In [2]:
%load_ext autoreload
%autoreload 2


# Dart scorer pipeline

This notebook exercises the classical dart-scoring algorithm from `examples/dart_scorer.py` in stages and then runs the real-image pipeline on a single selected board photo.


In [3]:
from pathlib import Path
import sys
import cv2
import numpy as np

cwd = Path.cwd().resolve()
repo_root = next(
    (path for path in (cwd, *cwd.parents) if (path / "pyproject.toml").is_file()),
    None,
)
if repo_root is None:
    raise RuntimeError("Run this notebook from the repo root or set repo_root manually.")
src_root = repo_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from dartcv.dart_scorer import (
    DartScorer,
    R_IB, R_OB, R_TI, R_TO, R_DI, R_DO,
    board_class,
    calibrate,
    polar,
    score_at,
    score_image,
)

ROOT = repo_root
IMAGE_PATH = ROOT / "data" / "images" / "single_dart" / "PXL_20261018_152634348.jpg"


## Stage 1: board geometry and scoring

Check the exact ring/sector scoring rules against points at known radii.


In [4]:
assert DartScorer.score_model_point(0, 0) == 50
assert DartScorer.score_model_point(10, 0) == 25
assert DartScorer.score_model_point(*polar(103, 0)) == 60
assert DartScorer.score_model_point(*polar(166, 0)) == 40
assert DartScorer.score_model_point(*polar(130, 18)) == 1
assert DartScorer.score_model_point(*polar(200, 0)) == 0

for point in [(0, 0), (10, 0), polar(103, 0), polar(166, 0), polar(130, 18)]:
    print(point, DartScorer.score_model_point(*point))


AssertionError: 

## Stage 2: synthetic board model and calibration

Generate a synthetic dartboard, derive the board class map and run the calibration helpers.


In [6]:
def synthetic_board(size=800):
    img = np.full((size, size, 3), 128, dtype=np.uint8)
    ys, xs = np.mgrid[0:size, 0:size]
    dx = (xs - size / 2) / 4.0
    dy = (ys - size / 2) / 4.0
    r = np.hypot(dx, dy)
    ang = (np.degrees(np.arctan2(dx, -dy)) + 9) % 360
    idx = (ang // 18).astype(int) % 20
    even = idx % 2 == 0
    base = np.where(even[..., None], (20, 20, 20), (235, 235, 235)).astype(np.uint8)
    ring = np.where(even[..., None], (40, 40, 210), (60, 160, 40)).astype(np.uint8)
    img[r <= R_DO] = base[r <= R_DO]
    for lo, hi in ((R_TI, R_TO), (R_DI, R_DO)):
        mask = (r > lo) & (r <= hi)
        img[mask] = ring[mask]
    img[r <= R_OB] = (60, 160, 40)
    img[r <= R_IB] = (40, 40, 210)
    return img

img = synthetic_board()
ys, xs = np.mgrid[0:img.shape[0], 0:img.shape[1]]
cx, cy = img.shape[1] / 2, img.shape[0] / 2
dx = (xs - cx) / 4.0
dy = (ys - cy) / 4.0
r = np.hypot(dx, dy)
theta = (np.degrees(np.arctan2(dx, -dy)) + 9) % 360
cls = board_class(r, theta)
G, agreement = calibrate(cls, cls.copy())
print({"agreement": agreement, "homography_shape": G.shape})


{'agreement': np.float32(0.8578786), 'homography_shape': (3, 3)}


## Stage 3: real-image pipeline run

Use the selected board image in `data/images/raw_board/IMG_2092.jpg` to run the full classical scorer pipeline for a smoke test.


In [16]:
IMAGE_PATH = ROOT / "data" / "images" / "single_dart" / "PXL_20261018_152634348.jpg"
IMAGE_PATH = ROOT / "data" / "images" / "single_dart" / "PXL_20261018_153340630.jpg"
IMAGE_PATH = ROOT / "data" / "images" / "multi_dart" / "PXL_20261018_154411671.jpg"

IMAGE_PATH = ROOT / "data" / "images" / "multi_dart" / "PXL_20261018_154459036.jpg"


In [20]:
assert IMAGE_PATH.exists(), f"missing image: {IMAGE_PATH}"
output_dir = ROOT / "out" / "dart_scorer_demo"
output_dir.mkdir(parents=True, exist_ok=True)
score_out = output_dir / IMAGE_PATH.name
total, darts, agreement = score_image(str(IMAGE_PATH), str(score_out), debug_dir=output_dir)

print({"total": total, "n_darts": len(darts), "agreement": agreement, "output": str(score_out)})


source=/Users/wueli/sw_source/dartcv/data/images/multi_dart/PXL_20261018_154459036.jpg
agreement=89.7%  darts=2  total=120
debug_dir=/Users/wueli/sw_source/dartcv/out/dart_scorer_demo
debug_metrics=/Users/wueli/sw_source/dartcv/out/dart_scorer_demo/debug_metrics.json
{'total': 120, 'n_darts': 2, 'agreement': np.float32(0.8973512), 'output': '/Users/wueli/sw_source/dartcv/out/dart_scorer_demo/PXL_20261018_154459036.jpg'}


## Notes

This pipeline is a classic CV smoke test, not a guarantee of perfect detection on all photos. The board model and calibration stages are intentionally explicit and easy to inspect in the notebook.
